In [12]:
# Import necessary libraries
import os
import sys
# Add the utils folder to the system path so we can import custom modules
sys.path.append(os.path.abspath(os.path.join("../..")))
import matplotlib.pyplot as plt
import torch
import numpy as np
import torch.nn as nn
import pandas as pd
from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data, feature_names

# Set the plotting style for better-looking visualizations
plt.style.available
plt.style.use('seaborn-v0_8')

In [13]:
for seed in [42, 7, 21]:
    print(f"\n========== Running Seed {seed} ==========")
    seed_everything(seed)

    # ---------------- LOAD DATA ----------------
    train_loader, val_loader, pipeline = load_data(
        train_path="../../datasets_challenge/train",
        train_ratio=0.98,
        batchsize=64,
        columns_to_drop=["sideSlip","vyCG","time","longitude","latitude"],
        sequence_length=20,
        threshold=True,
        threshold_value=3.0,
        threshold_column="vxCG",
        target="sideSlip",
        scaling=True,
    )

    test_loader = load_test_data(
        test_path="../../datasets_challenge/test",
        batchsize=64,
        pipeline=pipeline,
        load_anonymized_data=True
    )

    # ---------------- MODEL ----------------
    class RNN(nn.Module):
        """
        Recurrent Neural Network for sideslip angle prediction.
        
        Architecture:
        1. RNN Layer - Processes sequential input
        2. Fully Connected Layer - Transforms RNN output
        3. Output Layer - Produces final prediction
        
        Dropout layers are included to prevent overfitting.
        """
        def __init__(self, input_size, output_size=1, rnn_units=28, fc_units=28, 
                    activation_fct=nn.ReLU(), dropout=[0.0, 0, 0.00], rnn_nonlinearity='relu'):
            super(RNN, self).__init__()
            
            # RNN layer: processes the sequence and captures temporal dependencies
            self.rnn_layer = nn.RNN(input_size=input_size, hidden_size=rnn_units, 
                                    num_layers=1, nonlinearity=rnn_nonlinearity, batch_first=True)
            
            # Fully connected layers for processing RNN output
            self.fc = nn.Linear(rnn_units, fc_units)
            self.output_layer = nn.Linear(fc_units, output_size)
            
            # Dropout layers to prevent overfitting (randomly zeros out neurons during training)
            self.dropout1 = nn.Dropout(dropout[0])  # Applied to input
            self.dropout2 = nn.Dropout(dropout[1])  # Applied after RNN
            self.dropout3 = nn.Dropout(dropout[2])  # Applied after FC layer
            
            self.activation_fct = activation_fct
    
        def forward(self, x):
            # x shape: (batch_size, sequence_length, input_size)
            x = self.dropout1(x)
            
            # Process sequence through RNN
            x, _ = self.rnn_layer(x)  # Output shape: (batch_size, sequence_length, rnn_units)
            
            # Take only the last time step's output (t0)
            x = x[:, -1, :]  # Shape: (batch_size, rnn_units)
            
            x = self.dropout2(x)
            x = self.activation_fct(self.fc(x))  # Apply fully connected layer with activation
            x = self.dropout3(x)
            x = self.output_layer(x)  # Final prediction
            
            return x
        
    # ---------------- MODEL ----------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = RNN(
        input_size=len(feature_names),
        rnn_units=48,
        fc_units=48,
        dropout=[0,0,0]
    ).to(device)

    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    # ---------------- TRAIN WITH EARLY STOP ----------------
    best_loss = float("inf")
    patience = 8
    wait = 0

    for epoch in range(60):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

        # -------- validation --------
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                val_loss += criterion(model(X), y).item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch+1}  Val Loss: {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
            torch.save(model.state_dict(), f"model_seed_{seed}.pth")
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping!")
                break

    # ---------------- LOAD BEST MODEL ----------------
    model.load_state_dict(torch.load(f"model_seed_{seed}.pth"))
    model.eval()

    # ---------------- TEST PREDICTION ----------------
    preds = []
    with torch.no_grad():
        for X, _ in test_loader:
            X = X.to(device)
            pred = model(X)
            preds.append(pred.cpu().numpy())

    preds = np.concatenate(preds).reshape(-1)
    print("Shape of Predictions:", preds.shape)

    # ---------------- SAVE ----------------
    df = pd.DataFrame({
        "predicted_sideslip": preds,
        "id": range(len(preds))
    })
    df.to_csv(f"submission_seed_{seed}.csv", index=False)
    print(f"Saved submission_seed_{seed}.csv")


========== Running Seed 42 ==========
../../datasets_challenge/train\train_part_001.csv
../../datasets_challenge/train\train_part_002.csv
../../datasets_challenge/train\train_part_003.csv
../../datasets_challenge/train\train_part_004.csv
../../datasets_challenge/train\train_part_005.csv
../../datasets_challenge/train\train_part_006.csv
../../datasets_challenge/train\train_part_007.csv
../../datasets_challenge/train\train_part_008.csv
../../datasets_challenge/train\train_part_009.csv
../../datasets_challenge/train\train_part_010.csv
../../datasets_challenge/train\train_part_011.csv
../../datasets_challenge/train\train_part_012.csv
../../datasets_challenge/train\train_part_013.csv
../../datasets_challenge/train\train_part_014.csv
../../datasets_challenge/train\train_part_015.csv
../../datasets_challenge/train\train_part_016.csv
../../datasets_challenge/train\train_part_017.csv
../../datasets_challenge/train\train_part_018.csv
../../datasets_challenge/train\train_part_019.csv
../../datas

c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([3])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([53])) that is different to the input size (torch.Size([53, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same siz

Epoch 1  Val Loss: 0.8717
Epoch 2  Val Loss: 0.8718
Epoch 3  Val Loss: 0.8714
Epoch 4  Val Loss: 0.8711
Epoch 5  Val Loss: 0.8717
Epoch 6  Val Loss: 0.8707
Epoch 7  Val Loss: 0.8713
Epoch 8  Val Loss: 0.8724
Epoch 9  Val Loss: 0.8713
Epoch 10  Val Loss: 0.8708
Epoch 11  Val Loss: 0.8708
Epoch 12  Val Loss: 0.8712
Epoch 13  Val Loss: 0.8713
Epoch 14  Val Loss: 0.8716
Early stopping!
Shape of Predictions: (49328,)
Saved submission_seed_42.csv

========== Running Seed 7 ==========
../../datasets_challenge/train\train_part_001.csv
../../datasets_challenge/train\train_part_002.csv
../../datasets_challenge/train\train_part_003.csv
../../datasets_challenge/train\train_part_004.csv
../../datasets_challenge/train\train_part_005.csv
../../datasets_challenge/train\train_part_006.csv
../../datasets_challenge/train\train_part_007.csv
../../datasets_challenge/train\train_part_008.csv
../../datasets_challenge/train\train_part_009.csv
../../datasets_challenge/train\train_part_010.csv
../../datasets_ch

c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([3])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([53])) that is different to the input size (torch.Size([53, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same siz

Epoch 1  Val Loss: 0.8718
Epoch 2  Val Loss: 0.8718
Epoch 3  Val Loss: 0.8715
Epoch 4  Val Loss: 0.8711
Epoch 5  Val Loss: 0.8717
Epoch 6  Val Loss: 0.8707
Epoch 7  Val Loss: 0.8713
Epoch 8  Val Loss: 0.8725
Epoch 9  Val Loss: 0.8713
Epoch 10  Val Loss: 0.8708
Epoch 11  Val Loss: 0.8708
Epoch 12  Val Loss: 0.8712
Epoch 13  Val Loss: 0.8713
Epoch 14  Val Loss: 0.8716
Early stopping!
Shape of Predictions: (49328,)
Saved submission_seed_7.csv

========== Running Seed 21 ==========
../../datasets_challenge/train\train_part_001.csv
../../datasets_challenge/train\train_part_002.csv
../../datasets_challenge/train\train_part_003.csv
../../datasets_challenge/train\train_part_004.csv
../../datasets_challenge/train\train_part_005.csv
../../datasets_challenge/train\train_part_006.csv
../../datasets_challenge/train\train_part_007.csv
../../datasets_challenge/train\train_part_008.csv
../../datasets_challenge/train\train_part_009.csv
../../datasets_challenge/train\train_part_010.csv
../../datasets_ch

c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([3])) that is different to the input size (torch.Size([3, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
c:\mait_g2_drift_challenge\drift\Lib\site-packages\torch\nn\modules\loss.py:132: UserWarning: Using a target size (torch.Size([53])) that is different to the input size (torch.Size([53, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same siz

Epoch 1  Val Loss: 0.8717
Epoch 2  Val Loss: 0.8717
Epoch 3  Val Loss: 0.8714
Epoch 4  Val Loss: 0.8711
Epoch 5  Val Loss: 0.8717
Epoch 6  Val Loss: 0.8706
Epoch 7  Val Loss: 0.8712
Epoch 8  Val Loss: 0.8724
Epoch 9  Val Loss: 0.8713
Epoch 10  Val Loss: 0.8708
Epoch 11  Val Loss: 0.8707
Epoch 12  Val Loss: 0.8712
Epoch 13  Val Loss: 0.8713
Epoch 14  Val Loss: 0.8715
Early stopping!
Shape of Predictions: (49328,)
Saved submission_seed_21.csv
